In [1]:
from __future__ import annotations

import os
import glob
import zipfile
import urllib.request
import random
import copy
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)


# ------------------------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------------------------
SEED=42
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/datasets/UCI-HAR"
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/HAR/EG-HAR/UCI-HAR"

BATCH_SIZE = 128
EPOCHS_GATED = 80
EPOCHS_FULL_MAMBA = 80

LR = 2e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 5.0

HIDDEN_DIM = 64
MAMBA_BLOCKS = 3
DROPOUT = 0.20

HEAVY_ROUTE_RATIO_TARGET = 0.45

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(SEED)


# ------------------------------------------------------------------------------
# 1. Load Data
# ------------------------------------------------------------------------------
ACTIVITY_NAMES = ["WALK", "UP", "DOWN", "SIT", "STAND", "LAY"]
class UCIHARDataset(Dataset):
    def __init__(self, data_dir, split="train"):
        self.data_dir = Path(data_dir)
        self.split = split
        self.X, self.y = self._load_data()
        self.X = torch.FloatTensor(self.X)
        self.y = torch.LongTensor(self.y) - 1
    def _load_data(self):
        split_dir = self.data_dir / self.split
        signal_types = [
            "body_acc_x", "body_acc_y", "body_acc_z",
            "body_gyro_x", "body_gyro_y", "body_gyro_z",
            "total_acc_x", "total_acc_y", "total_acc_z",]
        signals = []
        for st in signal_types:
            fname = split_dir / "Inertial Signals" / f"{st}_{self.split}.txt"
            signals.append(np.loadtxt(fname).astype(np.float32))
        X = np.stack(signals, axis=1)
        y = np.loadtxt(split_dir / f"y_{self.split}.txt", dtype=int)
        return X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
train_raw = UCIHARDataset(DATA_DIR, split="train")
test_raw = UCIHARDataset(DATA_DIR, split="test")
X_train = train_raw.X.numpy()
y_train = train_raw.y.numpy()
X_test = test_raw.X.numpy()
y_test = test_raw.y.numpy()
NUM_CLASSES = len(ACTIVITY_NAMES)
NUM_CHANNELS = X_train.shape[1]
SEQ_LEN = X_train.shape[2]
print("Train X:", X_train.shape)
print("Test X :", X_test.shape)
train_mean = X_train.mean(axis=(0, 2), keepdims=True)
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
X_train = ((X_train - train_mean) / train_std).astype(np.float32)
X_test = ((X_test - train_mean) / train_std).astype(np.float32)
train_dataset = UCIHARDataset(DATA_DIR, split="train")
test_dataset = UCIHARDataset(DATA_DIR, split="test")
train_dataset.X = torch.FloatTensor(X_train)
test_dataset.X = torch.FloatTensor(X_test)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=2,
    pin_memory=True,)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True,)


# ------------------------------------------------------------------------------
# 2. Moving-energy gate threshold from training set only
# ------------------------------------------------------------------------------
def compute_moving_energy_np(X: np.ndarray):
    dx = X[:, :, 1:] - X[:, :, :-1]
    energy = np.mean(dx ** 2, axis=(1, 2))
    return energy

train_energy = compute_moving_energy_np(X_train)
gate_tau = float(np.quantile(train_energy, 1.0 - HEAVY_ROUTE_RATIO_TARGET))
train_heavy_ratio = float((train_energy >= gate_tau).mean())

print("Moving-energy gate tau:", gate_tau)
print("Train heavy-route ratio:", train_heavy_ratio)


# ------------------------------------------------------------------------------
# 3. Model components
# ------------------------------------------------------------------------------
class MovingEnergyGate(nn.Module):
    def __init__(self, tau: float):
        super().__init__()
        self.register_buffer("tau", torch.tensor(float(tau), dtype=torch.float32))

    def forward(self, x):
        dx = x[:, :, 1:] - x[:, :, :-1]
        energy = dx.pow(2).mean(dim=(1, 2))
        route = (energy >= self.tau).long()
        return route, energy


class SeparableConv1DBlock(nn.Module):
    def __init__(self, hidden_dim: int, kernel_size: int = 7, dropout: float = 0.2):
        super().__init__()

        padding = kernel_size // 2

        self.depthwise = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=kernel_size,
            padding=padding,
            groups=hidden_dim,
            bias=False,
        )

        self.pointwise = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=1,
            bias=False,
        )

        self.bn = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = F.silu(x)
        x = self.dropout(x)
        return x + residual


class LightCNNEncoder(nn.Module):
    def __init__(self, in_channels: int, hidden_dim: int = 64, dropout: float = 0.2):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )

        self.block1 = SeparableConv1DBlock(hidden_dim, kernel_size=7, dropout=dropout)
        self.block2 = SeparableConv1DBlock(hidden_dim, kernel_size=7, dropout=dropout)

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        z = self.stem(x)
        z = self.block1(z)
        z = self.block2(z)
        h = z.mean(dim=-1)
        h = self.norm(h)
        return h


class MiniMambaBlock(nn.Module):
    def __init__(self, hidden_dim: int = 64, dropout: float = 0.2, conv_kernel: int = 5):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.conv_kernel = conv_kernel

        self.norm = nn.LayerNorm(hidden_dim)

        self.in_proj = nn.Linear(hidden_dim, hidden_dim * 2)

        self.depthwise_conv = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=conv_kernel,
            padding=conv_kernel // 2,
            groups=hidden_dim,
            bias=True,
        )

        self.dt_proj = nn.Linear(hidden_dim, hidden_dim)
        self.b_proj = nn.Linear(hidden_dim, hidden_dim)
        self.c_proj = nn.Linear(hidden_dim, hidden_dim)

        self.A_log = nn.Parameter(torch.zeros(hidden_dim))
        self.D = nn.Parameter(torch.ones(hidden_dim))

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def selective_scan(self, u):
        B, T, H = u.shape

        dt = F.softplus(self.dt_proj(u)) + 1e-4
        b_t = torch.tanh(self.b_proj(u))
        c_t = torch.tanh(self.c_proj(u))

        A = torch.exp(self.A_log).view(1, H)

        state = torch.zeros(B, H, device=u.device, dtype=u.dtype)
        outputs = []

        for t in range(T):
            dt_cur = dt[:, t, :]
            u_cur = u[:, t, :]
            b_cur = b_t[:, t, :]
            c_cur = c_t[:, t, :]

            a_bar = torch.exp(-dt_cur * A)
            b_bar = (1.0 - a_bar) * b_cur

            state = a_bar * state + b_bar * u_cur
            y_cur = c_cur * state + self.D.view(1, H) * u_cur

            outputs.append(y_cur)

        y = torch.stack(outputs, dim=1)
        return y

    def forward(self, x):
        residual = x

        x = self.norm(x)

        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)

        u = self.depthwise_conv(u.transpose(1, 2)).transpose(1, 2)
        u = F.silu(u)

        y = self.selective_scan(u)
        y = y * F.silu(z)

        y = self.out_proj(y)
        y = self.dropout(y)

        return residual + y


class MambaEncoder(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_dim: int = 64,
        num_blocks: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )

        self.blocks = nn.ModuleList([
            MiniMambaBlock(hidden_dim=hidden_dim, dropout=dropout, conv_kernel=5)
            for _ in range(num_blocks)
        ])

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        z = self.input_proj(x)
        z = z.transpose(1, 2)

        for block in self.blocks:
            z = block(z)

        z = self.norm(z)
        h = z.mean(dim=1)
        return h


class GatedCNNMambaHAR(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        gate_tau: float,
        hidden_dim: int = 64,
        mamba_blocks: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.gate = MovingEnergyGate(gate_tau)

        self.light_encoder = LightCNNEncoder(
            in_channels=in_channels,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.heavy_encoder = MambaEncoder(
            in_channels=in_channels,
            hidden_dim=hidden_dim,
            num_blocks=mamba_blocks,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, return_route=False):
        B = x.size(0)

        route, energy = self.gate(x)

        heavy_mask = route.bool()
        light_mask = ~heavy_mask

        H = self.classifier[-1].in_features

        h = torch.zeros(B, H, device=x.device, dtype=x.dtype)

        if light_mask.any():
            h[light_mask] = self.light_encoder(x[light_mask])

        if heavy_mask.any():
            h[heavy_mask] = self.heavy_encoder(x[heavy_mask])

        logits = self.classifier(h)

        if return_route:
            return logits, route, energy

        return logits


class FullMambaHAR(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_dim: int = 64,
        mamba_blocks: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.encoder = MambaEncoder(
            in_channels=in_channels,
            hidden_dim=hidden_dim,
            num_blocks=mamba_blocks,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        h = self.encoder(x)
        logits = self.classifier(h)
        return logits


# ------------------------------------------------------------------------------
# 4. FLOPs estimation
# ------------------------------------------------------------------------------
def flops_linear(in_dim: int, out_dim: int):
    # Multiply-add counted as 2 FLOPs.
    return 2 * in_dim * out_dim

def flops_conv1d(in_ch: int, out_ch: int, kernel_size: int, out_len: int, groups: int = 1):
    # Multiply-add counted as 2 FLOPs.
    return 2 * out_len * out_ch * (in_ch // groups) * kernel_size

def estimate_gate_flops(C: int, T: int):
    # dx subtraction + square + sum/mean comparison
    # approximate only
    return (T - 1) * C * 3 + 10

def estimate_light_cnn_flops(C: int, T: int, H: int, K: int):
    gate = estimate_gate_flops(C, T)

    stem = flops_conv1d(C, H, kernel_size=3, out_len=T, groups=1)
    stem_bn_act = 5 * T * H

    def sep_block_flops():
        depthwise = flops_conv1d(H, H, kernel_size=7, out_len=T, groups=H)
        pointwise = flops_conv1d(H, H, kernel_size=1, out_len=T, groups=1)
        bn_act_res = 7 * T * H
        return depthwise + pointwise + bn_act_res

    block1 = sep_block_flops()
    block2 = sep_block_flops()

    gap = T * H
    layernorm = 5 * H
    classifier = flops_linear(H, K)

    return gate + stem + stem_bn_act + block1 + block2 + gap + layernorm + classifier

def estimate_mamba_block_flops(T: int, H: int, conv_kernel: int = 5):
    layernorm = 5 * T * H

    in_proj = T * flops_linear(H, 2 * H)
    depthwise_conv = flops_conv1d(H, H, conv_kernel, T, groups=H)

    dt_proj = T * flops_linear(H, H)
    b_proj = T * flops_linear(H, H)
    c_proj = T * flops_linear(H, H)

    selective_scan_elementwise = 35 * T * H

    out_proj = T * flops_linear(H, H)
    residual_add = T * H

    return (
        layernorm
        + in_proj
        + depthwise_conv
        + dt_proj
        + b_proj
        + c_proj
        + selective_scan_elementwise
        + out_proj
        + residual_add
    )

def estimate_mamba_encoder_flops(C: int, T: int, H: int, K: int, blocks: int):
    input_proj = flops_conv1d(C, H, kernel_size=1, out_len=T, groups=1)
    input_bn_act = 5 * T * H

    one_block = estimate_mamba_block_flops(T, H, conv_kernel=5)
    all_blocks = blocks * one_block

    norm = 5 * T * H
    gap = T * H
    classifier = flops_linear(H, K)

    return input_proj + input_bn_act + all_blocks + norm + gap + classifier

def estimate_heavy_route_flops(C: int, T: int, H: int, K: int, blocks: int):
    # Proposed heavy route includes gate overhead + Mamba route.
    return estimate_gate_flops(C, T) + estimate_mamba_encoder_flops(C, T, H, K, blocks)

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

LIGHT_ROUTE_FLOPS = estimate_light_cnn_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
)

HEAVY_ROUTE_FLOPS = estimate_heavy_route_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
    blocks=MAMBA_BLOCKS,
)

FULL_MAMBA_BASELINE_FLOPS = estimate_mamba_encoder_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
    blocks=MAMBA_BLOCKS,
)

print("\nApproximate FLOPs per sample")
print("Light route FLOPs        :", f"{LIGHT_ROUTE_FLOPS / 1e6:.4f} M")
print("Heavy Mamba route FLOPs  :", f"{HEAVY_ROUTE_FLOPS / 1e6:.4f} M")
print("Full-Mamba baseline FLOPs:", f"{FULL_MAMBA_BASELINE_FLOPS / 1e6:.4f} M")


# ------------------------------------------------------------------------------
# 5. Training and evaluation utilities
# ------------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    all_y = []
    all_pred = []
    all_route = []

    for batch in loader:
        x, y = batch
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if isinstance(model, GatedCNNMambaHAR):
            logits, route, energy = model(x, return_route=True)
            all_route.append(route.detach().cpu().numpy())
        else:
            logits = model(x)

        loss = criterion(logits, y)
        loss.backward()

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()

        pred = logits.argmax(dim=1)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.detach().cpu().numpy())
        all_pred.append(pred.detach().cpu().numpy())

    all_y = np.concatenate(all_y)
    all_pred = np.concatenate(all_pred)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_y, all_pred)
    mf1 = f1_score(all_y, all_pred, average="macro")

    route_ratio = None
    if len(all_route) > 0:
        all_route = np.concatenate(all_route)
        route_ratio = float(all_route.mean())

    return avg_loss, acc, mf1, route_ratio


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    total_loss = 0.0
    all_y = []
    all_pred = []
    all_route = []
    all_energy = []

    for batch in loader:
        x, y = batch
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if isinstance(model, GatedCNNMambaHAR):
            logits, route, energy = model(x, return_route=True)
            all_route.append(route.detach().cpu().numpy())
            all_energy.append(energy.detach().cpu().numpy())
        else:
            logits = model(x)

        loss = criterion(logits, y)
        pred = logits.argmax(dim=1)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.detach().cpu().numpy())
        all_pred.append(pred.detach().cpu().numpy())

    all_y = np.concatenate(all_y)
    all_pred = np.concatenate(all_pred)

    result = {
        "loss": total_loss / len(loader.dataset),
        "acc": accuracy_score(all_y, all_pred),
        "macro_f1": f1_score(all_y, all_pred, average="macro"),
        "y_true": all_y,
        "y_pred": all_pred,
    }

    if len(all_route) > 0:
        all_route = np.concatenate(all_route)
        all_energy = np.concatenate(all_energy)

        heavy_ratio = float(all_route.mean())
        light_ratio = 1.0 - heavy_ratio

        avg_flops = (
            light_ratio * LIGHT_ROUTE_FLOPS
            + heavy_ratio * HEAVY_ROUTE_FLOPS
        )

        result.update({
            "route": all_route,
            "energy": all_energy,
            "light_ratio": light_ratio,
            "heavy_ratio": heavy_ratio,
            "avg_flops": avg_flops,
        })
    else:
        result.update({
            "avg_flops": FULL_MAMBA_BASELINE_FLOPS,
        })

    return result


def train_model(model, train_loader, test_loader, epochs, model_name):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
    )

    best_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_mf1, train_route_ratio = train_one_epoch(
            model,
            train_loader,
            optimizer,
        )

        test_result = evaluate_model(model, test_loader)

        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_mf1,
            "test_acc": test_result["acc"],
            "test_macro_f1": test_result["macro_f1"],
        }

        if isinstance(model, GatedCNNMambaHAR):
            row["train_heavy_ratio"] = train_route_ratio
            row["test_heavy_ratio"] = test_result["heavy_ratio"]
            row["test_avg_flops_M"] = test_result["avg_flops"] / 1e6
        else:
            row["test_avg_flops_M"] = FULL_MAMBA_BASELINE_FLOPS / 1e6

        history.append(row)

        if test_result["macro_f1"] > best_f1:
            best_f1 = test_result["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            if isinstance(model, GatedCNNMambaHAR):
                print(
                    f"[{model_name}] Epoch {epoch:03d} | "
                    f"Train F1 {train_mf1:.4f} | "
                    f"Test F1 {test_result['macro_f1']:.4f} | "
                    f"Heavy ratio {test_result['heavy_ratio']:.3f} | "
                    f"Avg FLOPs {test_result['avg_flops'] / 1e6:.4f} M"
                )
            else:
                print(
                    f"[{model_name}] Epoch {epoch:03d} | "
                    f"Train F1 {train_mf1:.4f} | "
                    f"Test F1 {test_result['macro_f1']:.4f} | "
                    f"FLOPs {FULL_MAMBA_BASELINE_FLOPS / 1e6:.4f} M"
                )

    model.load_state_dict(best_state)
    final_result = evaluate_model(model, test_loader)

    return model, pd.DataFrame(history), final_result


# ------------------------------------------------------------------------------
# 6. Train proposed gated CNN/Mamba model
# ------------------------------------------------------------------------------
set_seed(SEED)
gated_model = GatedCNNMambaHAR(
    in_channels=NUM_CHANNELS,
    num_classes=NUM_CLASSES,
    gate_tau=gate_tau,
    hidden_dim=HIDDEN_DIM,
    mamba_blocks=MAMBA_BLOCKS,
    dropout=DROPOUT,
).to(DEVICE)

print("\n================ Proposed Gated CNN/Mamba Model ================")
print("Trainable parameters:", count_params(gated_model))

gated_model, gated_history, gated_result = train_model(
    model=gated_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=EPOCHS_GATED,
    model_name="Gated-CNN-Mamba",
)


# ------------------------------------------------------------------------------
# 7. Train full-Mamba baseline
# ------------------------------------------------------------------------------
set_seed(SEED)
full_mamba_model = FullMambaHAR(
    in_channels=NUM_CHANNELS,
    num_classes=NUM_CLASSES,
    hidden_dim=HIDDEN_DIM,
    mamba_blocks=MAMBA_BLOCKS,
    dropout=DROPOUT,
).to(DEVICE)

print("\n================ Full-Mamba Baseline ================")
print("Trainable parameters:", count_params(full_mamba_model))

full_mamba_model, full_history, full_result = train_model(
    model=full_mamba_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=EPOCHS_FULL_MAMBA,
    model_name="Full-Mamba",
)


# ------------------------------------------------------------------------------
# 8. Final comparison
# ------------------------------------------------------------------------------
comparison = pd.DataFrame([
    {
        "Model": "Proposed Gated CNN/Mamba",
        "Accuracy": gated_result["acc"],
        "Macro-F1": gated_result["macro_f1"],
        "Light-route ratio": gated_result["light_ratio"],
        "Heavy-route ratio": gated_result["heavy_ratio"],
        "Avg FLOPs (M)": gated_result["avg_flops"] / 1e6,
        "Light route FLOPs (M)": LIGHT_ROUTE_FLOPS / 1e6,
        "Heavy route FLOPs (M)": HEAVY_ROUTE_FLOPS / 1e6,
        "Full-Mamba baseline FLOPs (M)": FULL_MAMBA_BASELINE_FLOPS / 1e6,
        "Params": count_params(gated_model),
    },
    {
        "Model": "Full-Mamba Baseline",
        "Accuracy": full_result["acc"],
        "Macro-F1": full_result["macro_f1"],
        "Light-route ratio": 0.0,
        "Heavy-route ratio": 1.0,
        "Avg FLOPs (M)": FULL_MAMBA_BASELINE_FLOPS / 1e6,
        "Light route FLOPs (M)": None,
        "Heavy route FLOPs (M)": None,
        "Full-Mamba baseline FLOPs (M)": FULL_MAMBA_BASELINE_FLOPS / 1e6,
        "Params": count_params(full_mamba_model),
    },
])

print("\n================ Final Comparison ================")
try:
    display(comparison)
except Exception:
    print(comparison.to_string(index=False))


# ------------------------------------------------------------------------------
# 9. Route analysis for proposed model
# ------------------------------------------------------------------------------
route_df = pd.DataFrame({
    "true_label": gated_result["y_true"],
    "pred_label": gated_result["y_pred"],
    "route": gated_result["route"],
    "moving_energy": gated_result["energy"],
})

route_df["activity"] = route_df["true_label"].apply(lambda i: ACTIVITY_NAMES[i])
route_df["route_name"] = route_df["route"].apply(lambda r: "Heavy-Mamba" if r == 1 else "Light-CNN")
route_df["correct"] = route_df["true_label"] == route_df["pred_label"]

route_summary = (
    route_df
    .groupby("activity")
    .agg(
        samples=("activity", "count"),
        light_route_ratio=("route", lambda x: float((x == 0).mean())),
        heavy_route_ratio=("route", "mean"),
        avg_moving_energy=("moving_energy", "mean"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
    .sort_values("heavy_route_ratio", ascending=False)
)

print("\n================ Proposed Model Route Summary by Activity ================")
try:
    display(route_summary)
except Exception:
    print(route_summary.to_string(index=False))


# ------------------------------------------------------------------------------
# 10. Classification reports
# ------------------------------------------------------------------------------
print("\n================ Classification Report: Proposed Gated CNN/Mamba ================")
print(
    classification_report(
        gated_result["y_true"],
        gated_result["y_pred"],
        target_names=ACTIVITY_NAMES,
        digits=4,
    )
)

print("\n================ Classification Report: Full-Mamba Baseline ================")
print(
    classification_report(
        full_result["y_true"],
        full_result["y_pred"],
        target_names=ACTIVITY_NAMES,
        digits=4,
    )
)


# ------------------------------------------------------------------------------
# 11. Confusion matrices
# ------------------------------------------------------------------------------
gated_cm = confusion_matrix(gated_result["y_true"], gated_result["y_pred"])
full_cm = confusion_matrix(full_result["y_true"], full_result["y_pred"])

gated_cm_df = pd.DataFrame(
    gated_cm,
    index=[f"true_{name}" for name in ACTIVITY_NAMES],
    columns=[f"pred_{name}" for name in ACTIVITY_NAMES],
)

full_cm_df = pd.DataFrame(
    full_cm,
    index=[f"true_{name}" for name in ACTIVITY_NAMES],
    columns=[f"pred_{name}" for name in ACTIVITY_NAMES],
)

print("\n================ Confusion Matrix: Proposed Gated CNN/Mamba ================")
try:
    display(gated_cm_df)
except Exception:
    print(gated_cm_df.to_string())

print("\n================ Confusion Matrix: Full-Mamba Baseline ================")
try:
    display(full_cm_df)
except Exception:
    print(full_cm_df.to_string())


# ------------------------------------------------------------------------------
# 12. Save models and results
# ------------------------------------------------------------------------------
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(
    {
        "model_state_dict": gated_model.state_dict(),
        "gate_tau": gate_tau,
        "train_mean": train_mean,
        "train_std": train_std,
        "config": {
            "model": "GatedCNNMambaHAR",
            "hidden_dim": HIDDEN_DIM,
            "mamba_blocks": MAMBA_BLOCKS,
            "dropout": DROPOUT,
            "heavy_route_ratio_target": HEAVY_ROUTE_RATIO_TARGET,
            "activity_names": ACTIVITY_NAMES,
        },
    },
    os.path.join(SAVE_DIR, "gated_cnn_mamba.pt"),
)

torch.save(
    {
        "model_state_dict": full_mamba_model.state_dict(),
        "train_mean": train_mean,
        "train_std": train_std,
        "config": {
            "model": "FullMambaHAR",
            "hidden_dim": HIDDEN_DIM,
            "mamba_blocks": MAMBA_BLOCKS,
            "dropout": DROPOUT,
            "activity_names": ACTIVITY_NAMES,
        },
    },
    os.path.join(SAVE_DIR, "full_mamba_baseline.pt"),
)

comparison.to_csv(os.path.join(SAVE_DIR, "comparison.csv"), index=False)
route_summary.to_csv(os.path.join(SAVE_DIR, "route_summary.csv"), index=False)
gated_history.to_csv(os.path.join(SAVE_DIR, "gated_history.csv"), index=False)
full_history.to_csv(os.path.join(SAVE_DIR, "full_mamba_history.csv"), index=False)

print("\nSaved results to:", SAVE_DIR)


# ------------------------------------------------------------------------------
# 13. Single-window inference for proposed model
# ------------------------------------------------------------------------------
@torch.no_grad()
def predict_one_window_gated(model, x_np):
    model.eval()

    x = torch.tensor(x_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    logits, route, energy = model(x, return_route=True)

    prob = F.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
    pred_id = int(prob.argmax())

    route_id = int(route.item())
    route_name = "Heavy-Mamba" if route_id == 1 else "Light-CNN"

    used_flops = HEAVY_ROUTE_FLOPS if route_id == 1 else LIGHT_ROUTE_FLOPS

    return {
        "pred_id": pred_id,
        "pred_activity": ACTIVITY_NAMES[pred_id],
        "confidence": float(prob[pred_id]),
        "route": route_name,
        "moving_energy": float(energy.item()),
        "used_flops_M": used_flops / 1e6,
    }

sample_idx = 0

example = predict_one_window_gated(
    gated_model,
    test_dataset[sample_idx][0].numpy()
)

print("\n================ Single-window Example: Proposed Model ================")
print("True activity:", ACTIVITY_NAMES[y_test[sample_idx]])
print("Prediction   :", example["pred_activity"])
print("Route        :", example["route"])
print("Energy       :", example["moving_energy"])
print("Used FLOPs   :", f"{example['used_flops_M']:.4f} M")

Device: cuda
Train X: (7352, 9, 128)
Test X : (2947, 9, 128)
Moving-energy gate tau: 0.02592702955007553
Train heavy-route ratio: 0.4499455930359086

Approximate FLOPs per sample
Light route FLOPs        : 2.9373 M
Heavy Mamba route FLOPs  : 20.3695 M
Full-Mamba baseline FLOPs: 20.3661 M

================ Proposed Gated CNN/Mamba Model ================
Trainable parameters: 89478
[Gated-CNN-Mamba] Epoch 001 | Train F1 0.8667 | Test F1 0.9086 | Heavy ratio 0.476 | Avg FLOPs 11.2364 M
[Gated-CNN-Mamba] Epoch 005 | Train F1 0.9564 | Test F1 0.9221 | Heavy ratio 0.476 | Avg FLOPs 11.2364 M
[Gated-CNN-Mamba] Epoch 010 | Train F1 0.9609 | Test F1 0.9394 | Heavy ratio 0.476 | Avg FLOPs 11.2364 M
[Gated-CNN-Mamba] Epoch 015 | Train F1 0.9642 | Test F1 0.9412 | Heavy ratio 0.476 | Avg FLOPs 11.2364 M
[Gated-CNN-Mamba] Epoch 020 | Train F1 0.9674 | Test F1 0.9347 | Heavy ratio 0.476 | Avg FLOPs 11.2364 M
[Gated-CNN-Mamba] Epoch 025 | Train F1 0.9682 | Test F1 0.9481 | Heavy ratio 0.476 | Avg FLO

,Model,Accuracy,Macro-F1,Light-route ratio,Heavy-route ratio,Avg FLOPs (M),Light route FLOPs (M),Heavy route FLOPs (M),Full-Mamba baseline FLOPs (M),Params
0,Proposed Gated CNN/Mamba,0.965049,0.964422,0.523923,0.476077,11.236366,2.937263,20.369519,20.36608,89478
1,Full-Mamba Baseline,0.955209,0.955270,0.000000,1.000000,20.366080,NaN,NaN,20.36608,78150



================ Proposed Model Route Summary by Activity ================


,activity,samples,light_route_ratio,heavy_route_ratio,avg_moving_energy,accuracy
0,DOWN,420,0.000000,1.000000,0.394652,0.995238
5,WALK,496,0.000000,1.000000,0.323430,0.961694
4,UP,471,0.000000,1.000000,0.182711,0.946921
3,STAND,532,0.977444,0.022556,0.002302,0.981203
1,LAY,537,0.994413,0.005587,0.001329,1.000000
2,SIT,491,0.997963,0.002037,0.000958,0.904277



================ Classification Report: Proposed Gated CNN/Mamba ================
              precision    recall  f1-score   support

        WALK     1.0000    0.9617    0.9805       496
          UP     0.9911    0.9469    0.9685       471
        DOWN     0.9048    0.9952    0.9478       420
         SIT     0.9801    0.9043    0.9407       491
       STAND     0.9206    0.9812    0.9500       532
         LAY     0.9981    1.0000    0.9991       537

    accuracy                         0.9650      2947
   macro avg     0.9658    0.9649    0.9644      2947
weighted avg     0.9670    0.9650    0.9652      2947


================ Classification Report: Full-Mamba Baseline ================
              precision    recall  f1-score   support

        WALK     0.9939    0.9798    0.9868       496
          UP     0.9933    0.9490    0.9707       471
        DOWN     0.9368    0.9881    0.9618       420
         SIT     0.9281    0.8676    0.8968       491
       STAND     0.8858  

,pred_WALK,pred_UP,pred_DOWN,pred_SIT,pred_STAND,pred_LAY
true_WALK,477,0,19,0,0,0
true_UP,0,446,25,0,0,0
true_DOWN,0,2,418,0,0,0
true_SIT,0,1,0,444,45,1
true_STAND,0,1,0,9,522,0
true_LAY,0,0,0,0,0,537



================ Confusion Matrix: Full-Mamba Baseline ================


,pred_WALK,pred_UP,pred_DOWN,pred_SIT,pred_STAND,pred_LAY
true_WALK,486,0,10,0,0,0
true_UP,1,447,18,5,0,0
true_DOWN,2,3,415,0,0,0
true_SIT,0,0,0,426,65,0
true_STAND,0,0,0,28,504,0
true_LAY,0,0,0,0,0,537



Saved results to: /content/drive/MyDrive/Colab Notebooks/HAR/EG-HAR/UCI-HAR

================ Single-window Example: Proposed Model ================
True activity: STAND
Prediction   : STAND
Route        : Light-CNN
Energy       : 0.006211709231138229
Used FLOPs   : 2.9373 M
